# HyperKT Objective 01 — Stage 6: End-to-end fine-tuning

**Runtime:** Google Colab → **T4 GPU** (L4 optional).  
**Inputs:** `hyperkt_stage2.zip`, `hyperkt_stage3b.zip`, `hyperkt_stage4.zip`, `hyperkt_stage5.zip`.  
**Output:** `hyperkt_stage6.zip`.

This experiment fine-tunes the Stage 5 mean prediction head and temporal/hypergraph fusion backbone together, selects checkpoints using validation AUC, and **does not read the test split**. The Beta concentration head is excluded from optimization: binary response likelihood does not identify it. This notebook records the exact validation protocol and checkpoints; it does not guarantee improved performance.

**Run all cells in order.** The initial checkpoint must match the saved architecture and vocabulary.

In [ ]:
import os, io, json, zipfile, copy, random, shutil, math
import numpy as np, pandas as pd, torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
from google.colab import files
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:',DEVICE,'PyTorch:',torch.__version__)
assert DEVICE.type=='cuda','Choose Runtime > Change runtime type > T4 GPU'


In [ ]:
uploaded=files.upload()
required=['hyperkt_stage2.zip','hyperkt_stage3b.zip','hyperkt_stage4.zip','hyperkt_stage5.zip']
assert all(x in uploaded for x in required), f'Missing: {[x for x in required if x not in uploaded]}'
z2,z3,z4,z5=[zipfile.ZipFile(io.BytesIO(uploaded[x])) for x in required]
def load_npz(z,name):
    with np.load(io.BytesIO(z.read(name)),allow_pickle=False) as f:
        return {k:f[k].copy() for k in f.files}
train=load_npz(z2,'train_sequences.npz');val=load_npz(z2,'val_sequences.npz')
hg=load_npz(z3,'hypergraph_repaired.npz')
assert hg['assignments'].shape==(149,32)
assert set(train['user_id']).isdisjoint(set(val['user_id']))
assert train['mask'].sum()==222562 and val['mask'].sum()==23858
print('Train targets:',int(train['mask'].sum()),'Validation targets:',int(val['mask'].sum()))


In [ ]:
PAD=1;N_SKILLS=151;MAXLEN=200
A=np.zeros((N_SKILLS,32),dtype=np.float32)
A[hg['skill_ids'].astype(int)]=hg['assignments'].astype(np.float32)
A[PAD]=0;A=torch.tensor(A)
def dataset(d):
    names=['history_skill','history_correct','target_skill','target_correct','mask']
    return TensorDataset(*(torch.as_tensor(d[k],dtype=torch.bool if k=='mask' else torch.long) for k in names))
train_loader=DataLoader(dataset(train),batch_size=16,shuffle=True,num_workers=0)
val_loader=DataLoader(dataset(val),batch_size=16,shuffle=False,num_workers=0)
class HyperKTTemporal(nn.Module):
    def __init__(self,use_hypergraph=False,d=256,heads=8,layers=4,dropout=.1):
        super().__init__();self.use_hypergraph=use_hypergraph
        self.skill=nn.Embedding(N_SKILLS,d,padding_idx=PAD)
        self.response=nn.Embedding(2,d);self.pos=nn.Embedding(MAXLEN,d)
        self.decay_raw=nn.Parameter(torch.tensor(-2.97))
        if use_hypergraph:
            self.register_buffer('assignments',A.clone())
            self.hg_proj=nn.Linear(32,d,bias=False);self.hg_gate=nn.Linear(2*d,d)
        enc=nn.TransformerEncoderLayer(d_model=d,nhead=heads,dim_feedforward=4*d,dropout=dropout,batch_first=True,activation='gelu',norm_first=True)
        self.encoder=nn.TransformerEncoder(enc,num_layers=layers,enable_nested_tensor=False)
        self.norm=nn.LayerNorm(d)
        self.head=nn.Sequential(nn.Linear(2*d,d),nn.GELU(),nn.Dropout(dropout),nn.Linear(d,1))
    def features(self,hs,hc,ts,mask):
        B,L=hs.shape;pos=torch.arange(L,device=hs.device)
        age=(L-1-pos).float();decay=torch.exp(-F.softplus(self.decay_raw)*age/MAXLEN)
        x=self.skill(hs)+self.response(hc.clamp(0,1))+self.pos(pos)[None,:,:]*decay[None,:,None]
        if self.use_hypergraph:
            structure=self.hg_proj(self.assignments[hs]);gate=torch.sigmoid(self.hg_gate(torch.cat([x,structure],dim=-1)))
            x=x+gate*structure
        causal=torch.triu(torch.ones(L,L,device=hs.device,dtype=torch.bool),diagonal=1)
        x=x.masked_fill(~mask.unsqueeze(-1),0.0)
        h=self.norm(self.encoder(x,mask=causal))
        target=self.skill(ts)
        if self.use_hypergraph:target=target+self.hg_proj(self.assignments[ts])
        return torch.cat([h,target],dim=-1)
    def forward(self,hs,hc,ts,mask):return self.head(self.features(hs,hc,ts,mask)).squeeze(-1)


In [ ]:
# Load the exact Stage 5 model weights, then unfreeze the temporal backbone.
ckpt=torch.load(io.BytesIO(z5.read('stage5_head.pt')),map_location='cpu',weights_only=True)
print('Checkpoint keys:',list(ckpt.keys()))


In [ ]:
class BetaHyperKT(nn.Module):
    def __init__(self,backbone):
        super().__init__();self.backbone=backbone
        self.mean_head=copy.deepcopy(backbone.head)
        self.concentration_head=nn.Sequential(nn.Linear(512,128),nn.GELU(),nn.Dropout(.1),nn.Linear(128,1))
    def forward(self,hs,hc,ts,mask):
        f=self.backbone.features(hs,hc,ts,mask)
        mu=torch.sigmoid(self.mean_head(f).squeeze(-1))
        kappa=2.0+F.softplus(self.concentration_head(f).squeeze(-1))
        return mu,kappa
model=BetaHyperKT(HyperKTTemporal(use_hypergraph=True)).to(DEVICE)
model.load_state_dict(ckpt['state_dict'],strict=True)
for p in model.backbone.parameters():p.requires_grad_(True)
# The original backbone output head is unused by BetaHyperKT.forward.
for p in model.backbone.head.parameters():p.requires_grad_(False)
# No supervised information identifies Beta concentration from single binary labels.
for p in model.concentration_head.parameters():p.requires_grad_(False)
trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
print('Trainable parameters:',trainable)
optimizer=torch.optim.AdamW([
    {'params':[p for p in model.backbone.parameters() if p.requires_grad], 'lr':2e-5},
    {'params':[p for p in model.mean_head.parameters() if p.requires_grad], 'lr':1e-4},
],weight_decay=0.01)


In [ ]:
from sklearn.metrics import roc_auc_score,log_loss,brier_score_loss
import pandas as pd

def ece(y,p,n_bins=15):
    edges=np.linspace(0,1,n_bins+1);ids=np.clip(np.searchsorted(edges,p,side='right')-1,0,n_bins-1)
    return float(sum((ids==i).mean()*abs(y[ids==i].mean()-p[ids==i].mean()) for i in range(n_bins) if np.any(ids==i)))
def metrics(y,p):
    p=np.clip(p,1e-7,1-1e-7)
    return {'auc':float(roc_auc_score(y,p)),'nll':float(log_loss(y,p,labels=[0,1])),
            'brier':float(brier_score_loss(y,p)),'ece_15':ece(y,p),'n_targets':int(len(y))}
def epoch_train():
    model.train();total=0.;count=0
    for hs,hc,ts,y,m in train_loader:
        hs,hc,ts,y,m=[v.to(DEVICE) for v in (hs,hc,ts,y,m)]
        optimizer.zero_grad(set_to_none=True)
        mu,_=model(hs,hc,ts,m)
        loss=F.binary_cross_entropy(mu[m].clamp(1e-6,1-1e-6),y[m].float())
        loss.backward();nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],1.0)
        optimizer.step();count+=int(m.sum());total+=float(loss.detach())*int(m.sum())
    return total/count
@torch.no_grad()
def predict(loader,mc=1):
    model.train(mc>1)
    yy=[];pp=[];vv=[]
    for hs,hc,ts,y,m in loader:
        hs,hc,ts,y,m=[v.to(DEVICE) for v in (hs,hc,ts,y,m)]
        probs=[]
        for _ in range(mc):
            mu,_=model(hs,hc,ts,m)
            probs.append(mu[m].detach().cpu().numpy())
        probs=np.stack(probs)
        yy.append(y[m].cpu().numpy());pp.append(probs.mean(axis=0));vv.append(probs.var(axis=0))
    return np.concatenate(yy),np.concatenate(pp),np.concatenate(vv)


In [ ]:
# Initial validation before fine-tuning; keeps a baseline in this same run.
y0,p0,_=predict(val_loader,mc=1)
initial=metrics(y0,p0)
print('Initial validation:',initial)


In [ ]:
EPOCHS=12;PATIENCE=4;best_auc=initial['auc'];best_epoch=0
best_state=copy.deepcopy({k:v.detach().cpu() for k,v in model.state_dict().items()})
history=[{'epoch':0,'train_nll':None,**{'val_'+k:v for k,v in initial.items()}}]
wait=0
for epoch in range(1,EPOCHS+1):
    train_loss=epoch_train()
    y,p,_=predict(val_loader,mc=1)
    v=metrics(y,p)
    history.append({'epoch':epoch,'train_nll':train_loss,**{'val_'+k:x for k,x in v.items()}})
    print(f'Epoch {epoch}: train NLL={train_loss:.5f}, val AUC={v["auc"]:.6f}, val ECE={v["ece_15"]:.5f}')
    if v['auc']>best_auc+1e-5:
        best_auc=v['auc'];best_epoch=epoch
        best_state=copy.deepcopy({k:t.detach().cpu() for k,t in model.state_dict().items()});wait=0
    else:
        wait+=1
        if wait>=PATIENCE:
            print('Early stopping');break
model.load_state_dict(best_state)
print('Selected epoch:',best_epoch,'Best validation AUC:',best_auc)


In [ ]:
# Validation-only inference with 30 MC passes. No test data used.
y,p_det,_=predict(val_loader,mc=1)
y2,p_mc,var_mc=predict(val_loader,mc=30)
assert np.array_equal(y,y2) and len(y)==23858
report={'stage':6,'seed':SEED,'selected_epoch':best_epoch,'initial_validation':initial,
        'selected_deterministic_validation':metrics(y,p_det),
        'selected_mc30_validation':metrics(y,p_mc),
        'mc30_variance_mean':float(var_mc.mean()),
        'test_used':False,
        'notes':'Student-disjoint validation; selection on deterministic validation AUC. MC dropout is approximate uncertainty, not calibrated Beta concentration. Hypergraph assignments are fixed, fusion weights fine-tuned.'}
print(json.dumps(report,indent=2))


In [ ]:
import matplotlib.pyplot as plt
out='/content/hyperkt_stage6';os.makedirs(out,exist_ok=True)
torch.save({'state_dict':best_state,'selected_epoch':best_epoch,'seed':SEED,'model':'BetaHyperKT_stage6'},out+'/stage6_model.pt')
pd.DataFrame(history).to_csv(out+'/training_history.csv',index=False)
with open(out+'/report.json','w') as f:json.dump(report,f,indent=2)
np.savez_compressed(out+'/validation_predictions.npz',y=y,p_deterministic=p_det,p_mc30=p_mc,mc30_variance=var_mc)
plt.figure(figsize=(7,4));plt.plot([h['epoch'] for h in history],[h['val_auc'] for h in history],marker='o');plt.xlabel('Epoch');plt.ylabel('Validation ROC-AUC');plt.title('Stage 6 validation performance');plt.grid(alpha=.25);plt.tight_layout();plt.savefig(out+'/validation_auc.png',dpi=180);plt.close()
with open(out+'/protocol.txt','w') as f:
    f.write('Stage 6 end-to-end fine-tuning from Stage 5 checkpoint. Frozen Stage 3B hypergraph assignments, trainable temporal and fusion backbone, trainable mean head. Validation-only selection. No test set loaded. Concentration head excluded from optimization. MC30 evaluated after selection.\n')
shutil.make_archive('/content/hyperkt_stage6','zip',out)
print('Saved /content/hyperkt_stage6.zip')
files.download('/content/hyperkt_stage6.zip')


## Interpretation

Compare Stage 6 against Stage 5 validation AUC **0.758921 (MC30)** and Stage 4 fusion **0.757777**. These comparisons are exploratory because validation data is repeatedly used for model development. **Do not report final test AUC yet.** After the model and evaluation protocol are frozen, use independent held-out test data and repeated seeds. The stage 6 checkpoint does not train the hypergraph incidence matrix itself; only its downstream fusion is fine-tuned.